# Probe 023 launcher (phase C)
Colab is a compute worker only. This driver kernel NEVER imports the model stack -- `run.py` runs as a child process, so no kernel restart is ever needed.

**One-time setup:** create a fine-grained GitHub PAT scoped to this single repository, Contents: Read and write, with an expiry. In Colab: key icon (Secrets) -> add `SCOUT_RESULTS_PAT` -> enable notebook access. The PAT never appears in this notebook or its output.

Results branch (contract-bound): `results/probe-023-349af5ad0b3e`

Per session: run all cells top to bottom. After a disconnect, rerun all cells -- run.py resumes from the bundle on Drive, and the transport cell pushes whatever is new.

In [ ]:
PHASE = 'C'
REPO_URL = 'https://github.com/Moroseui/concept-research-scout.git'
PIN_COMMIT = 'd388e23f5a791382fc059e89c3de509d50da2080'
RESULTS_BRANCH = 'results/probe-023-349af5ad0b3e'
OUTPUT_DIR = '/content/drive/MyDrive/concept-research-scout-results/023_v2'

In [ ]:
from google.colab import drive, userdata
import os
drive.mount('/content/drive')
GH_PAT = userdata.get('SCOUT_RESULTS_PAT')  # never printed
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')  # inherited by the run.py child; never printed

In [ ]:
!rm -rf /content/scout-repo
!git clone {REPO_URL} /content/scout-repo
%cd /content/scout-repo
!git checkout {PIN_COMMIT}

In [ ]:
!pip install -q -r probes/023/requirements.txt

In [ ]:
# --- Generated staging: Zenodo record 16731717 (Drive-persistent, idempotent) ---
import os, json, urllib.request
STAGE = '/content/drive/MyDrive/staging-16731717'
RECORD_JSON = STAGE + '/zenodo_record.json'
DATA_DIR = STAGE + '/extracted'
os.makedirs(STAGE, exist_ok=True)
if not os.path.exists(RECORD_JSON):
    with urllib.request.urlopen('https://zenodo.org/api/records/16731717') as r:
        rec = json.load(r)
    assert str(rec['id']) != '16731717', 'resolved to the concept record; need an immutable child version'
    json.dump(rec, open(RECORD_JSON, 'w'), indent=2)
rec = json.load(open(RECORD_JSON))
_a = [f for f in rec['files'] if f['key'].endswith('.7z')]
assert len(_a) == 1, _a
ARCHIVE = STAGE + '/' + _a[0]['key']
ARCHIVE_URL = _a[0]['links']['self']
print('pinned record', rec['id'], _a[0]['key'], round(_a[0]['size']/1e9, 1), 'GB')

In [ ]:
!wget -c -O "{ARCHIVE}" "{ARCHIVE_URL}"

In [ ]:
SUFFIXES = ['_space-ncct_cbf.nii.gz', '_space-ncct_cbv.nii.gz', '_space-ncct_mtt.nii.gz', '_space-ncct_tmax.nii.gz', '_lesion-msk.nii.gz', '_ncct.nii.gz']
if not os.path.isdir(DATA_DIR):
    !apt-get -qq install -y p7zip-full
    _inc = ' '.join('-ir!*' + x for x in SUFFIXES)
    !7z x "{ARCHIVE}" -o"{DATA_DIR}" {_inc} -y
!find "{DATA_DIR}" -type f | wc -l

In [ ]:
# Console (incl. any crash traceback) persists to Drive; refresh-proof.
!mkdir -p {OUTPUT_DIR}
!python probes/023/run.py --phase {PHASE} --output-dir {OUTPUT_DIR} --data-dir {DATA_DIR} --archive-file {ARCHIVE} --record-json {RECORD_JSON} 2>&1 | tee -a {OUTPUT_DIR}/driver_console.log

In [ ]:
# E1 transport: mirror the bundle onto the contract-bound results
# branch. ORDER MATTERS: check out the branch FIRST, then overlay the
# bundle (copy-then-checkout fails after session 1: git refuses to
# overwrite untracked files the branch already tracks). The PAT rides
# in a header, never in argv or output.
import shutil, subprocess, pathlib, base64, datetime
repo = pathlib.Path('/content/scout-repo')
dest = repo / 'probes/023/results_v2'
def git(*a, **k):
    r = subprocess.run(['git', *a], cwd=repo, capture_output=True, text=True, **k)
    if r.returncode: raise SystemExit(f'git {a[0]} failed: {r.stderr[-400:]}')
    return r.stdout
git('config', 'user.email', 'colab-runner@scout.local')
git('config', 'user.name', 'scout colab runner')
auth = base64.b64encode(f'x-access-token:{GH_PAT}'.encode()).decode()
hdr = f'http.extraheader=AUTHORIZATION: basic {auth}'
if subprocess.run(['git', '-c', hdr, 'fetch', 'origin', RESULTS_BRANCH], cwd=repo, capture_output=True).returncode == 0:
    git('checkout', '-B', RESULTS_BRANCH, f'origin/{RESULTS_BRANCH}')
else:
    git('checkout', '-B', RESULTS_BRANCH, PIN_COMMIT)
if dest.exists(): shutil.rmtree(dest)
shutil.copytree(OUTPUT_DIR, dest)
git('add', '-f', 'probes/023/results_v2')
stamp = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')
subprocess.run(['git', 'commit', '-m', f'session results {stamp}'], cwd=repo, capture_output=True)
git('-c', hdr, 'push', 'origin', RESULTS_BRANCH)
print('pushed', RESULTS_BRANCH)

When `run.py` reports the study complete, the results-validate workflow on the pushed branch verifies the bundle and opens the record-result PR. Merging that PR is the human gate.